# Vaidya — Phase 2: QLoRA Training (Google Colab / Colab Enterprise)

**Before running:**
1. Runtime → Change runtime type → select GPU
2. Left sidebar → 🔑 Secrets → add and enable notebook access for:
   - `AWS_ACCESS_KEY_ID`
   - `AWS_SECRET_ACCESS_KEY`
   - `AWS_DEFAULT_REGION` → `ap-south-1`

**Expected runtime:**
- A100 80GB (Colab Enterprise): ~1.5–2 hrs — full epoch
- L4 24GB (Colab Enterprise): ~2.5 hrs — full epoch
- T4 16GB (free Colab): ~11 hrs — use `--max-steps 500`

> `train.py` auto-detects Colab and loads AWS credentials from secrets automatically.

## Step 1 — Install packages

In [ ]:
!pip install -q \
    transformers==4.47.0 \
    peft==0.14.0 \
    trl==0.15.2 \
    bitsandbytes==0.45.3 \
    accelerate==0.34.2 \
    datasets==3.2.0 \
    mlflow==2.19.0 \
    boto3

print('✅ Install done')

## Step 2 — Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → GPU'

gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc   = torch.cuda.get_device_capability()

print(f'GPU  : {gpu}')
print(f'VRAM : {vram:.1f} GB')
print(f'CC   : {cc[0]}.{cc[1]}')
print(f'BF16 native : {cc[0] >= 8}')   # True on A100/H100/L4, False on T4/V100

if cc[0] < 8:
    print('⚠️  T4/V100 detected — training uses FP16. Step 7 adds --max-steps 500 automatically.')

## Step 3 — Pull training scripts from GitHub

In [ ]:
import os, shutil

os.chdir('/content')

!git clone --depth 1 https://github.com/Anand09-in/vaidya.git _repo 2>&1 | tail -2

for f in ['train.py', 'qlora_config.py']:
    shutil.copy(f'_repo/layer2_finetune/{f}', f'./{f}')
    print(f'✅  {f}')

!rm -rf _repo

In [ ]:
# Verify versions
import importlib.util

for fname, modname in [('train.py', 'train_mod'), ('qlora_config.py', 'qlora_config')]:
    spec = importlib.util.spec_from_file_location(modname, f'/content/{fname}')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    print(f"{fname}  →  version {getattr(mod, '__version__', 'MISSING')}")

## Step 4 — Verify AWS credentials

`train.py` loads credentials from Colab secrets automatically.  
This cell just verifies they work.

In [ ]:
from google.colab import userdata
import os, boto3

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = userdata.get('AWS_DEFAULT_REGION')

s3 = boto3.client('s3')
buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
print('Buckets:', buckets)
assert 'vaidya-artifacts' in buckets, '❌ vaidya-artifacts not found — check secrets'

## Step 5 — Pull data from S3

In [ ]:
import boto3, os, pandas as pd

os.makedirs('/content/data', exist_ok=True)
s3 = boto3.client('s3')

for page in s3.get_paginator('list_objects_v2').paginate(Bucket='vaidya-artifacts', Prefix='data/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith('.parquet'):
            fname = key.split('/')[-1]
            local = f'/content/data/{fname}'
            if not os.path.exists(local):
                print(f'Downloading {key}...')
                s3.download_file('vaidya-artifacts', key, local)

for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'/content/data/{split}.parquet')
    print(f'{split}: {len(df):,} rows')

## Step 6 — Pre-flight check

In [ ]:
import importlib

required = ['transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets', 'mlflow', 'boto3']
all_ok = True
for pkg in required:
    try:
        m = importlib.import_module(pkg)
        print(f'✅  {pkg} {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'❌  {pkg}  — {e}')
        all_ok = False

if not all_ok:
    print('\n⚠️  Re-run Step 1 and restart runtime.')
else:
    print('\n✅ Environment ready.')

## Step 7 — Run training

In [ ]:
import torch
cc = torch.cuda.get_device_capability()

# Auto-set step cap: T4 (CC < 8) needs cap to fit in session; A100/H100/L4 run full epoch
max_steps_flag = '--max-steps 500' if cc[0] < 8 else ''
print(f'GPU CC {cc[0]}.{cc[1]} → {"max_steps=500 (T4)" if cc[0] < 8 else "full epoch (no cap)"}')

In [ ]:
# ⚡ FULL TRAINING RUN
# A100: ~1.5–2 hrs | L4: ~2.5 hrs | T4: ~11 hrs (auto-capped to 500 steps)
# Change --run-name for subsequent runs

!python /content/train.py --run-name run1 {max_steps_flag}

## Step 8 — Verify checkpoint on S3

In [ ]:
import boto3

s3 = boto3.client('s3')

print('=== Checkpoint files ===')
resp = s3.list_objects_v2(Bucket='vaidya-artifacts', Prefix='checkpoints/run1/')
for obj in resp.get('Contents', []):
    print(f"  {obj['Key']}  ({obj['Size']/1e6:.1f} MB)")

print()
print('=== MLflow ===')
resp2 = s3.list_objects_v2(Bucket='vaidya-artifacts', Prefix='mlflow/')
for obj in resp2.get('Contents', []):
    print(f"  {obj['Key']}")

## Step 9 — Quick eval

In [ ]:
import json, glob

states = glob.glob('/content/checkpoints/run1/checkpoint-*/trainer_state.json')
if not states:
    states = glob.glob('/content/checkpoints/run1/trainer_state.json')

if states:
    with open(sorted(states)[-1]) as f:
        state = json.load(f)
    log_history = state.get('log_history', [])
    train_losses = [(e['step'], e['loss'])      for e in log_history if 'loss'      in e]
    eval_losses  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]
    if train_losses:
        print(f'Train loss — first: {train_losses[0][1]:.4f}  last: {train_losses[-1][1]:.4f}')
    if eval_losses:
        print(f'Eval  loss — first: {eval_losses[0][1]:.4f}  last: {eval_losses[-1][1]:.4f}')
else:
    print('No trainer_state.json — check /content/checkpoints/')

## Gate check — before moving to Phase 3

- ✅ Training loss decreased
- ✅ Eval loss decreased
- ✅ Checkpoint at `s3://vaidya-artifacts/checkpoints/run1/`
- ✅ MLflow db at `s3://vaidya-artifacts/mlflow/mlflow.db`

**Next:** Phase 3 — accuracy eval (base Mistral-7B vs fine-tuned on MedMCQA test set)